In [1]:
!wget https://iasl-btm.iis.sinica.edu.tw/BNER/Content/Revised_JNLPBA.zip

--2025-10-15 13:25:45--  https://iasl-btm.iis.sinica.edu.tw/BNER/Content/Revised_JNLPBA.zip
Resolving iasl-btm.iis.sinica.edu.tw (iasl-btm.iis.sinica.edu.tw)... 140.109.20.133
Connecting to iasl-btm.iis.sinica.edu.tw (iasl-btm.iis.sinica.edu.tw)|140.109.20.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2533317 (2.4M) [application/x-zip-compressed]
Saving to: ‘Revised_JNLPBA.zip’

Revised_JNLPBA.zip  100%[===================>]   2.42M  1.60MB/s    in 1.5s    

2025-10-15 13:25:47 (1.60 MB/s) - ‘Revised_JNLPBA.zip’ saved [2533317/2533317]



In [2]:
!unzip -n Revised_JNLPBA.zip -d Revised_JNLPBA

Archive:  Revised_JNLPBA.zip
  inflating: Revised_JNLPBA/Genia4EReval1.iob2  
  inflating: Revised_JNLPBA/Genia4EReval2.iob2  
  inflating: Revised_JNLPBA/Genia4ERtask1.iob2  
  inflating: Revised_JNLPBA/Genia4ERtask2.iob2  


In [3]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=1b0473ff386d9ba878df61bdd0007774eb2c4b7affaf841b16547f7eab0aa9ef
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval


In [4]:
from typing import List, Tuple
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from collections import Counter
from itertools import chain

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
import re  # Untuk pencocokan pola teks (regex)

# Fungsi untuk membaca file IOB dan mengubahnya jadi daftar kalimat dan label
def read_iob(file_path):
    sentences, labels = [], []   # Menyimpan kumpulan kalimat dan label
    tokens, tags = [], []        # Menyimpan token dan tag sementara
    header_pattern = re.compile(r"^###MEDLINE:\d+")  # Pola untuk melewati baris header

    with open(file_path, "r") as f:  # Membuka file dan membaca baris per baris
        for line in f:
            line = line.strip()
            if not line:  # Jika baris kosong → akhir kalimat
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens, tags = [], []
                continue

            if header_pattern.match(line):  # Lewati baris header
                continue

            parts = line.split()
            if len(parts) == 2:  # Pisahkan kata dan tag
                word, tag = parts
                tokens.append(word)
                tags.append(tag)

        # Simpan kalimat terakhir jika belum ditutup baris kosong
        if tokens:
            sentences.append(tokens)
            labels.append(tags)

    return sentences, labels  # Kembalikan list kalimat dan label


# Membaca empat file dataset JNLPBA (dua untuk training, dua untuk evaluasi)
train_sent1, train_labels1 = read_iob("Revised_JNLPBA/Genia4ERtask1.iob2")
train_sent2, train_labels2 = read_iob("Revised_JNLPBA/Genia4ERtask2.iob2")
eval_sent1, eval_labels1   = read_iob("Revised_JNLPBA/Genia4EReval1.iob2")
eval_sent2, eval_labels2   = read_iob("Revised_JNLPBA/Genia4EReval2.iob2")

# Menggabungkan data training dan evaluasi
train_sentences = train_sent1 + train_sent2
train_labels    = train_labels1 + train_labels2
test_sentences  = eval_sent1 + eval_sent2
test_labels     = eval_labels1 + eval_labels2


# Fungsi untuk menampilkan contoh token dan label
def print_example(tokens, tags, n=20):
    # Menampilkan n token pertama beserta labelnya
    for t, l in zip(tokens[:n], tags[:n]):
        print(f"{t:20} {l}")


print_example(train_sentences[0], train_labels[0])

IL-2                 B-DNA
gene                 I-DNA
expression           O
and                  O
NF-kappa             B-protein
B                    I-protein
activation           O
through              O
CD28                 B-protein
requires             O
reactive             O
oxygen               O
production           O
by                   O
5-lipoxygenase       B-protein
.                    O


# Preprocessing

In [6]:
print("=== Contoh Data Asli ===")
print(train_sentences[0])
print(train_labels[0])

=== Contoh Data Asli ===
['IL-2', 'gene', 'expression', 'and', 'NF-kappa', 'B', 'activation', 'through', 'CD28', 'requires', 'reactive', 'oxygen', 'production', 'by', '5-lipoxygenase', '.']
['B-DNA', 'I-DNA', 'O', 'O', 'B-protein', 'I-protein', 'O', 'O', 'B-protein', 'O', 'O', 'O', 'O', 'O', 'B-protein', 'O']


In [7]:
# Dataset khusus untuk Named Entity Recognition (NER)
# Mengubah data token dan label menjadi bentuk tensor agar bisa diproses oleh model PyTorch
class NERDataset(Dataset):
    def __init__(self, examples: List[Tuple[List[str], List[str]]],
                 word_vocab, char_vocab, label_vocab, max_word_len=30):
        # Menyimpan data dan kamus (vocabulary) untuk kata, karakter, dan label
        self.examples = examples
        self.wv = word_vocab
        self.cv = char_vocab
        self.lv = label_vocab
        self.max_word_len = max_word_len

    def __len__(self):
        # Mengembalikan jumlah contoh dalam dataset
        return len(self.examples)

    def token_to_id(self, token):
        # Mengubah token menjadi ID berdasarkan word_vocab
        # Jika token tidak ada di kamus → gunakan token UNK (unknown)
        return self.wv.get(token, self.wv[UNK])

    def chars_to_ids(self, token):
        # Mengubah setiap karakter dalam token menjadi ID berdasarkan char_vocab
        ids = [self.cv.get(ch, self.cv[UNK]) for ch in token[:self.max_word_len]]
        # Menambahkan padding jika panjang kata kurang dari max_word_len
        ids = ids + [self.cv[PAD]] * (self.max_word_len - len(ids))
        return ids

    def __getitem__(self, idx):
        # Mengambil satu contoh data berdasarkan indeks
        tokens, labels = self.examples[idx]
        # Ubah kata, karakter, dan label menjadi ID tensor
        w_ids = [self.token_to_id(t) for t in tokens]
        c_ids = [self.chars_to_ids(t) for t in tokens]  # bentuk: seq_len x max_word_len
        l_ids = [self.lv[l] for l in labels]
        # Kembalikan tensor beserta panjang sekuens
        return torch.tensor(w_ids, dtype=torch.long), \
               torch.tensor(c_ids, dtype=torch.long), \
               torch.tensor(l_ids, dtype=torch.long), \
               len(tokens)


# Fungsi untuk melakukan padding pada batch data agar bisa diproses bersamaan dalam model
def ner_collate(batch):
    # batch: list berisi tuple (w_ids, c_ids, l_ids, length)
    lengths = [b[3] for b in batch]
    max_len = max(lengths)  # panjang sekuens terpanjang
    max_word_len = batch[0][1].size(1)  # panjang maksimal kata (karakter)
    
    ws, cs, ls, masks = [], [], [], []
    for w_ids, c_ids, l_ids, L in batch:
        # Padding untuk kata (word-level)
        pad_w = torch.full((max_len - L,), 0, dtype=torch.long)  # PAD index 0
        ws.append(torch.cat([w_ids, pad_w], dim=0))

        # Padding untuk karakter (char-level)
        pad_c = torch.full((max_len - L, max_word_len), 0, dtype=torch.long)
        cs.append(torch.cat([c_ids, pad_c], dim=0))

        # Padding untuk label (dengan -100 agar diabaikan oleh loss function)
        pad_l = torch.full((max_len - L,), -100, dtype=torch.long)
        ls.append(torch.cat([l_ids, pad_l], dim=0))

        # Mask menandai posisi token valid (True) dan padding (False)
        masks.append(torch.cat([
            torch.ones(L, dtype=torch.bool),
            torch.zeros(max_len - L, dtype=torch.bool)
        ], dim=0))

    # Menggabungkan semua batch jadi tensor
    return torch.stack(ws), torch.stack(cs), torch.stack(ls), torch.stack(masks)

In [8]:
PAD = "<PAD>"   # Token khusus untuk padding
UNK = "<UNK>"   # Token khusus untuk kata tak dikenal (unknown)

# 1️⃣ Membuat Word Vocabulary
all_words = {w for s in train_sentences for w in s}
word_vocab = {w: i + 2 for i, w in enumerate(all_words)}  # reserve 0=PAD, 1=UNK
word_vocab["<PAD>"] = 0
word_vocab["<UNK>"] = 1

# 2️⃣ Membuat Character Vocabulary
all_chars = {ch for w in all_words for ch in w}
char_vocab = {c: i + 2 for i, c in enumerate(all_chars)}
char_vocab["<PAD>"] = 0
char_vocab["<UNK>"] = 1

# 3️⃣ Membuat Label Vocabulary
all_labels = {l for seq in train_labels for l in seq}
label_vocab = {l: i for i, l in enumerate(sorted(all_labels))}

# 4️⃣ Membungkus Data ke dalam Dataset PyTorch
train_examples = list(zip(train_sentences, train_labels))
test_examples  = list(zip(test_sentences, test_labels))

train_dataset = NERDataset(train_examples, word_vocab, char_vocab, label_vocab)
test_dataset  = NERDataset(test_examples, word_vocab, char_vocab, label_vocab)

# 5️⃣ Membuat DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=ner_collate)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=ner_collate)

# 6️⃣ Mengecek Bentuk Data Batch
for ws, cs, ls, masks in train_loader:
    print("Word IDs:", ws.shape)   
    print("Char IDs:", cs.shape)   
    print("Label IDs:", ls.shape)  
    print("Masks:", masks.shape)   
    break

Word IDs: torch.Size([32, 101])
Char IDs: torch.Size([32, 101, 30])
Label IDs: torch.Size([32, 101])
Masks: torch.Size([32, 101])


In [9]:
# Tampilkan hasil sesudah
sample_tokens = train_sentences[0]
sample_labels = train_labels[0]
print("=== Sebelum Preprocessing ===")
print("Tokens :", sample_tokens)
print("Labels :", sample_labels)

train_examples = list(zip(train_sentences, train_labels))
train_dataset = NERDataset(train_examples, word_vocab, char_vocab, label_vocab)
# Ubah jadi tensor dengan class NERDataset
w_ids, c_ids, l_ids, length = train_dataset[0]

# Tampilkan hasil sesudah
print("\n=== Sesudah Preprocessing ===")
print("Word IDs  :", w_ids.tolist())
print("Label IDs :", l_ids.tolist())
print("Char IDs per token (contoh 5 token pertama):")
for i, c in enumerate(c_ids[:5]):
    print(f" {sample_tokens[i]:15s} → {c.tolist()}")

=== Sebelum Preprocessing ===
Tokens : ['IL-2', 'gene', 'expression', 'and', 'NF-kappa', 'B', 'activation', 'through', 'CD28', 'requires', 'reactive', 'oxygen', 'production', 'by', '5-lipoxygenase', '.']
Labels : ['B-DNA', 'I-DNA', 'O', 'O', 'B-protein', 'I-protein', 'O', 'O', 'B-protein', 'O', 'O', 'O', 'O', 'O', 'B-protein', 'O']

=== Sesudah Preprocessing ===
Word IDs  : [13926, 17637, 13163, 13892, 17650, 12518, 11245, 10002, 17241, 3782, 13968, 4115, 12324, 5291, 17421, 13928]
Label IDs : [0, 5, 10, 10, 4, 9, 10, 10, 4, 10, 10, 10, 10, 10, 4, 10]
Char IDs per token (contoh 5 token pertama):
 IL-2            → [36, 77, 66, 71, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
 gene            → [50, 78, 22, 78, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
 expression      → [78, 62, 68, 19, 78, 13, 13, 81, 44, 22, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
 and             → [38, 22, 65, 0, 0, 0, 0, 0, 0,

In [10]:
class CharCNNWordBiLSTM(nn.Module):
    def __init__(self,
                 vocab_size,
                 char_vocab_size,
                 label_size,
                 word_emb_dim=200,
                 char_emb_dim=30,
                 char_out_channels=50,
                 char_kernel_size=3,
                 lstm_hidden=256,
                 lstm_layers=1,
                 dropout=0.3,
                 pretrained_word_emb=None):
        super().__init__()

        # 1. Membuat embedding untuk kata (word-level)
        #    Setiap kata diwakili oleh vektor berdimensi word_emb_dim.
        #    Jika ada embedding pretrained (misal GloVe), maka bobotnya diganti sesuai pretrained tersebut.
        self.word_emb = nn.Embedding(vocab_size, word_emb_dim, padding_idx=0)
        if pretrained_word_emb is not None:
            self.word_emb.weight.data[:pretrained_word_emb.shape[0]] = torch.tensor(pretrained_word_emb)

        # 2. Membuat embedding untuk karakter (char-level) dan CNN untuk ekstraksi fitur morfologi
        #    CNN di sini menangkap pola huruf seperti akhiran atau awalan yang sering muncul.
        self.char_emb = nn.Embedding(char_vocab_size, char_emb_dim, padding_idx=0)
        self.char_cnn = nn.Conv1d(in_channels=char_emb_dim,
                                  out_channels=char_out_channels,
                                  kernel_size=char_kernel_size,
                                  padding=1)

        # 3. Menyiapkan dropout untuk mencegah overfitting.
        #    input_dim = gabungan fitur dari word embedding + karakter CNN.
        self.dropout = nn.Dropout(dropout)
        input_dim = word_emb_dim + char_out_channels

        # 4. Membuat BiLSTM untuk menangkap konteks dua arah (dari kiri dan kanan).
        #    Dibagi dua karena BiLSTM punya dua arah, jadi tiap arah hanya punya setengah dimensi dari total hidden.
        self.bilstm = nn.LSTM(input_dim,
                              lstm_hidden // 2,
                              num_layers=lstm_layers,
                              batch_first=True,
                              bidirectional=True)

        # 5. Membuat lapisan klasifikasi linear untuk menghasilkan prediksi label per token.
        #    Output dari LSTM (konteksual) akan diproyeksikan ke jumlah kelas label.
        self.classifier = nn.Linear(lstm_hidden, label_size)

    def forward(self, w_ids, c_ids, mask=None):
        # 6. Embedding kata: mengubah indeks kata menjadi vektor kata
        #    Dimensi output: (batch_size, seq_len, word_emb_dim)
        bsz, seq_len = w_ids.size()
        word_emb = self.word_emb(w_ids)

        # 7. Embedding karakter + CNN + MaxPooling
        #    Setiap kata terdiri dari beberapa karakter → CNN mengekstrak pola antar huruf,
        #    kemudian max pooling mengambil fitur paling dominan untuk tiap kata.
        b_s, s_l, max_w = c_ids.size()
        ch = c_ids.view(-1, max_w)
        ch_emb = self.char_emb(ch)
        ch_emb = ch_emb.transpose(1, 2)
        ch_conv = self.char_cnn(ch_emb)
        ch_pool = torch.max(ch_conv, dim=2)[0]
        ch_pool = ch_pool.view(b_s, s_l, -1)

        # 8. Menggabungkan fitur dari word dan char level → lalu masuk ke BiLSTM
        #    Dropout diterapkan sebelum LSTM untuk regularisasi.
        x = torch.cat([word_emb, ch_pool], dim=2)
        x = self.dropout(x)
        lstm_out, _ = self.bilstm(x)

        # 9. Output dari BiLSTM diproyeksikan ke ruang label dengan Linear layer.
        #    Dropout sekali lagi untuk stabilisasi training.
        logits = self.classifier(self.dropout(lstm_out))
        return logits